# 03 · Signals — `gwmock-signal`

The signal engine, used directly from Python. Its pipeline is
**waveform → projection → injection**, and every stage is a function you
can call (and swap) yourself.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260706-leuven-gw-workshop-gwmock/blob/main/materials/notebooks/03-signals.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in the correlated-noise machinery used for the
# stochastic-background examples.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 1. Waveforms — `WaveformFactory`

The factory registers every time-domain approximant your LAL install
provides (plus optional PyCBC / JAX-ripple backends), and returns GWpy
`TimeSeries` polarizations:

In [ ]:
from gwmock_signal.waveform import WaveformFactory

factory = WaveformFactory()     # build once, reuse — construction enumerates approximants
models = factory.list_models()
print(len(models), "models available, e.g.", models[:5])

In [ ]:
pol = factory.generate(
    "IMRPhenomD",
    {
        "tc": 1_400_000_000.0,          # coalescence GPS time (required)
        "detector_frame_mass_1": 36.0,
        "detector_frame_mass_2": 29.0,
        "spin_1z": 0.0,
        "spin_2z": 0.0,
        "distance": 410.0,
        "inclination": 0.0,
        "coa_phase": 0.0,
    },
    sampling_frequency=4096.0,
    minimum_frequency=20.0,
)
hp, hx = pol["plus"], pol["cross"]
print("duration:", hp.duration)

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(hp.times.value - 1_400_000_000.0, hp.value, lw=0.7)
plt.xlabel("time since coalescence [s]")
plt.ylabel(r"$h_+$")
plt.title("GW150914-like IMRPhenomD waveform")
plt.show()

## 2. Projection — from polarizations to detectors

Antenna patterns + light-travel-time delays (optionally following Earth's
rotation). Works for LAL codes (`H1`, `L1`, `V1`, …) **and** the bundled
ET geometries:

In [ ]:
from gwmock_signal import Network
from gwmock_signal.projection import project_polarizations_to_network

net = Network.from_preset("ET-Triangle-Sardinia")
print("ET triangle detectors:", [d.name for d in net.detector_names])

proj = project_polarizations_to_network(
    pol, net.detector_names,
    right_ascension=1.375, declination=-1.211,
    polarization_angle=2.659, earth_rotation=True,
)

plt.figure(figsize=(9, 3))
for name, ts in proj.items():
    plt.plot(ts.times.value - 1_400_000_000.0, ts.value, lw=0.6, label=name)
plt.xlim(-0.35, 0.05)
plt.xlabel("time since coalescence [s]")
plt.ylabel("strain")
plt.legend(loc="upper left", fontsize=8)
plt.title("same waveform, three arms of the ET triangle")
plt.show()

## 3. One-call pipeline — `inject_cbc_signal`

Waveform + projection + injection into a background (here: none) in one
call. This is exactly what the `signal` block of a gwmock config drives:

In [ ]:
from gwmock_signal.pipeline import inject_cbc_signal

params = {
    "detector_frame_mass_1": 36.0, "detector_frame_mass_2": 29.0,
    "spin_1z": 0.0, "spin_2z": 0.0,
    "coa_time": 1126259462.4, "distance": 410.0,
    "right_ascension": 1.375, "declination": -1.211,
    "polarization_angle": 2.659, "inclination": 2.5, "coa_phase": 0.0,
}
net2l = Network.from_preset("ET-2L-Aligned")
stack = inject_cbc_signal("IMRPhenomD", params, net2l.detector_names, None,
                          sampling_frequency=2048.0, minimum_frequency=10.0)

print("detectors in stack:", stack.detector_names)
strains = stack.to_dict()
plt.figure(figsize=(9, 3))
for name, ts in strains.items():
    plt.plot(ts.times.value - 1126259462.4, ts.value, lw=0.6, label=name)
plt.xlim(-0.5, 0.1)
plt.xlabel("time since coalescence [s]")
plt.ylabel("strain")
plt.legend(fontsize=8)
plt.title("DetectorStrainStack for ET-2L")
plt.show()

## 4. Bring your own waveform

Anything returning `{"plus", "cross"}` GWpy series can be registered —
numerical relativity, ROMs, bursts. The contract is a keyword-only
callable:

In [ ]:
import numpy as np
from astropy import units as u
from gwpy.timeseries import TimeSeries


def toy_gaussian_sine_burst(*, waveform_model, tc, sampling_frequency,
                            minimum_frequency, **kwargs):
    width_s = float(kwargs.get("width", 0.1))
    f_hz = float(kwargs.get("f0", 150.0))
    dt = 1.0 / sampling_frequency
    n_half = max(8, int(width_s * sampling_frequency))
    t = (np.arange(-n_half, n_half + 1) * dt) + tc
    env = np.exp(-0.5 * ((t - tc) / (width_s / 3)) ** 2)
    phase = 2 * np.pi * f_hz * (t - tc)
    return {
        "plus": TimeSeries(env * np.cos(phase), t0=float(t[0]), dt=dt,
                           unit=u.dimensionless_unscaled),
        "cross": TimeSeries(env * np.sin(phase), t0=float(t[0]), dt=dt,
                            unit=u.dimensionless_unscaled),
    }


factory.register_model("toy_burst", toy_gaussian_sine_burst)
burst = factory.generate("toy_burst", {"tc": 1_400_000_000.0, "f0": 120.0, "width": 0.05},
                         sampling_frequency=4096.0, minimum_frequency=20.0)

plt.figure(figsize=(8, 2.5))
plt.plot(burst["plus"].times.value - 1_400_000_000.0, burst["plus"].value)
plt.xlabel("time since $t_c$ [s]")
plt.title("your own registered model")
plt.show()

## 5. Stochastic backgrounds — `StochasticBackgroundSimulator`

Not every signal is a transient. The SGWB simulator draws a **correlated**
Gaussian background across a network from a power-law
$\Omega_{\rm GW}(f)$. The correlation between detectors is set by the
**overlap-reduction function** of the pair's geometry — for the co-located
ET triangle it is *negative*:

In [ ]:
from gwmock_signal import StochasticBackgroundSimulator

sgwb = StochasticBackgroundSimulator(duration=16.0, seed=7)
bg = sgwb.simulate(
    {"omega_ref": 1e-9, "spectral_index": 0.0, "reference_frequency": 25.0},
    Network.from_preset("ET-Triangle-Sardinia").detector_names,
    sampling_frequency=512.0, minimum_frequency=5.0,
)
bgd = bg.to_dict()
names = list(bgd)
a, b = bgd[names[0]].value, bgd[names[1]].value
print("channels:", names)
r = np.corrcoef(a, b)[0, 1]
print(f"Pearson correlation {names[0]} vs {names[1]}: {r:.3f}")
print("(independent instrument noise would give ~0; co-located triangle arms")
print(" rotated by 120 deg are ANTI-correlated — overlap reduction -3/8 = -0.375)")

## ✏️ Try it (5 min)

1. Generate the same binary with `SEOBNRv4` and overlay it on the
   `IMRPhenomD` waveform. (Both in `factory.list_models()`?)
2. Face-on vs edge-on: regenerate with `inclination: 0.0` and
   `inclination: 1.57` — compare amplitudes.
3. Make the toy burst chirp: multiply `f_hz` by `(1 + 20 * (t - tc))`
   inside the phase.

Next: **04 · Noise** — the other half of realistic data.